# GraphPad Exports

This notebook is a lightweight orchestration layer for CSV exports intended for GraphPad Prism.

The goal is to keep GraphPad-facing exports separate from paper figure plotting:

1. For wall contacts per reward interval, use the scalar `.npz` files generated by Panel 21 in `paper_figure_panels.ipynb`.
2. For mean between-reward tortuosity, use the scalar `.npz` files generated by Panel 26 in `paper_figure_panels.ipynb`.
3. For percent time on agarose, run the upstream analyses for either the agarose HTL or flat HTL chamber here and then export pre-minus-post per-fly deltas directly to CSV.

Each command cell defaults to preview mode. Set the matching `RUN_...` toggle to `True` when you want to execute a stage.

## Setup

Run this once after opening or restarting the notebook kernel.

In [ ]:
%cd ..

In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Markdown, display

ROOT = Path.cwd()
print(f"Notebook working directory: {ROOT}")


def show_command_block(title, command):
    display(Markdown(f"**{title}**"))
    display(Markdown("```bash\n" + str(command) + "\n```"))


def show_output_path(path):
    artifact = ROOT / path
    if artifact.exists():
        display(Markdown(f"Output exists: `{artifact}`"))
    else:
        display(Markdown(f"Expected output not found yet: `{artifact}`"))


def run_stage(stage_name, commands, run=False):
    display(Markdown(f"### {stage_name}"))
    if not run:
        display(Markdown("_Preview mode only. Set this stage's toggle to `True` to execute._"))
        for item in commands:
            show_command_block(item["label"], item["command"])
            if item.get("output_path"):
                show_output_path(item["output_path"])
        return

    for item in commands:
        show_command_block(item["label"], item["command"])
        completed = subprocess.run(item["command"], shell=True, cwd=ROOT)
        if completed.returncode != 0:
            raise RuntimeError(
                f"Command failed for '{item['label']}' with exit code {completed.returncode}."
            )
        if item.get("output_path"):
            show_output_path(item["output_path"])


## Existing Panel Scalar Exports

These two GraphPad exports consume scalar `.npz` files that are already produced by the existing paper-panel workflow.

Before running this stage:

- Run **Panel 21 Data Export Commands** in `paper_figure_panels.ipynb` to generate the wall-contact `.npz` files.
- Run **Panel 26 Data Export Commands** in `paper_figure_panels.ipynb` to generate the tortuosity `.npz` files.

This stage only converts those existing per-fly scalar exports into GraphPad-friendly column CSV files.

In [ ]:
existing_panel_graphpad_exports = [
    {
        "label": "Panel 21 wall contacts per reward interval -> GraphPad CSV",
        "command": "python scripts/export_graphpad_csv.py scalar-npz --input 'Ctrl>Kir FLC=exports/wall_contact_per_rwd_interval_intact_ctrlKir_flat_lgc_T2_2026-04-20.npz' --input 'PFN>Kir FLC=exports/wall_contact_per_rwd_interval_intact_pfnKir_flat_lgc_T2_2026-04-20.npz' --input 'AR Ctrl>Kir FLC=exports/wall_contact_per_rwd_interval_ar_ctrlKir_flat_lgc_T2_2026-04-20.npz' --out exports/wall_contacts_per_reward_interval_graphpad.csv",
        "output_path": "exports/wall_contacts_per_reward_interval_graphpad.csv",
    },
    {
        "label": "Panel 26 mean between-reward tortuosity -> GraphPad CSV",
        "command": "python scripts/export_graphpad_csv.py scalar-npz --input 'Ctrl>Kir FLC=exports/tort_mean_fullPath_intact_ctrlKir_flatLgc_T2_inclWall_2026-04-29.npz' --input 'PFN>Kir FLC=exports/tort_mean_fullPath_intact_pfnKir_flatLgc_T2_inclWall_2026-04-29.npz' --input 'AR Ctrl>Kir FLC=exports/tort_mean_fullPath_ar_ctrlKir_flatLgc_T2_inclWall_2026-04-29.npz' --out exports/between_reward_tortuosity_mean_graphpad.csv",
        "output_path": "exports/between_reward_tortuosity_mean_graphpad.csv",
    },
]

RUN_EXISTING_PANEL_GRAPHPAD_EXPORTS = False
run_stage(
    "Existing panel scalar exports -> GraphPad CSV",
    existing_panel_graphpad_exports,
    run=RUN_EXISTING_PANEL_GRAPHPAD_EXPORTS,
)


## Percent Time On Agarose - Agarose HTL Chamber

This analysis is not a numbered paper panel, so the upstream export commands live here. This first configuration is for the **agarose HTL chamber**.

The first stage runs `analyze.py --agarose` for the relevant fly groups. `analyze.py` writes `learning_stats.csv` in the repository root; each command then copies that file to the canonical CSV path used by the final GraphPad export.

The control group is reused for both the upd3 KO and upd2+3 KO comparisons, so only one control `learning_stats.csv` is generated.

In [ ]:
agarose_htl_learning_stats_exports = [
    {
        "label": "Control learning_stats.csv for upd KO agarose-time analysis (agarose HTL)",
        "command": """mkdir -p '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO' && python analyze.py -v '/media/Synology4/Yang Chen/2025-02-0[12]/c5[12]_*,/media/Synology4/Yang Chen/2025-02-0[12]/c6[12]_*,/media/Synology4/Yang Chen/2025-02-02/c4[12]_*' -f 0-9 --rmCC 5 --agarose && cp learning_stats.csv '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.ctrl.agarose_htl.csv'""",
        "output_path": "/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.ctrl.agarose_htl.csv",
    },
    {
        "label": "upd3 KO learning_stats.csv for agarose-time analysis (agarose HTL)",
        "command": """mkdir -p '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO' && python analyze.py -v '/media/Synology4/Yang Chen/2025-01-06/c[12]_*,/media/Synology4/Yang Chen/2025-01-1[2678]/c5[12]_*,/media/Synology4/Yang Chen/2025-01-18/c2[12]_*' -f 0-9 --rmCC 5 --agarose && cp learning_stats.csv '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.mutant.agarose_htl.csv'""",
        "output_path": "/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.mutant.agarose_htl.csv",
    },
    {
        "label": "upd2+3 KO learning_stats.csv for agarose-time analysis (agarose HTL)",
        "command": """mkdir -p '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO' && python analyze.py -v "/media/Synology4/Yang Chen/2025-01-06/c2[12]_*,/media/Synology4/Yang Chen/2025-01-1[2678]/c6[12]_*,/media/Synology4/Yang Chen/2025-01-18/c[12]_*,/media/Synology4/Yang Chen/2025-01-19/c5[12]_*,/media/Synology4/Yang Chen/2025-01-20/c4[12]_*,/media/Synology4/Yang Chen/2025-01-20/c5[12]_*,/media/Synology4/Yang Chen/2025-01-2[01]/c6[12]_*" -f 0-9 --rmCC 5 --agarose && cp learning_stats.csv '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO/learning_stats.upd2+3_ko.mutant.agarose_htl.csv'""",
        "output_path": "/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO/learning_stats.upd2+3_ko.mutant.agarose_htl.csv",
    },
]

RUN_AGAROSE_HTL_LEARNING_STATS_EXPORTS = False
run_stage(
    "Agarose HTL learning_stats.csv exports",
    agarose_htl_learning_stats_exports,
    run=RUN_AGAROSE_HTL_LEARNING_STATS_EXPORTS,
)


### Agarose HTL GraphPad CSV

This final stage exports the per-fly reduction values used by the agarose-time plot:

`pre last 10m - T3 post last 10m`

The values are percentage-point reductions in percent time spent on agarose.

In [ ]:
agarose_htl_graphpad_exports = [
    {
        "label": "Percent time on agarose reduction (agarose HTL) -> GraphPad CSV",
        "command": "python scripts/export_graphpad_csv.py agarose-time --group Control=/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.ctrl.agarose_htl.csv --group 'upd3 KO=/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.mutant.agarose_htl.csv' --group 'upd2+3 KO=/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO/learning_stats.upd2+3_ko.mutant.agarose_htl.csv' --section '% time over agarose (contact events begin when body center crosses agarose border)' --out exports/agarose_time_reduction_agarose_htl_graphpad.csv",
        "output_path": "exports/agarose_time_reduction_agarose_htl_graphpad.csv",
    },
]

RUN_AGAROSE_HTL_GRAPHPAD_EXPORT = False
run_stage(
    "Agarose HTL reduction values -> GraphPad CSV",
    agarose_htl_graphpad_exports,
    run=RUN_AGAROSE_HTL_GRAPHPAD_EXPORT,
)


## Percent Time On Agarose - Flat HTL Chamber

This configuration repeats the same analysis for the **flat HTL chamber**, using chamber-qualified intermediate files so that these results remain separate from the agarose HTL exports.

The analysis parameters and final pre-minus-post calculation are identical to the agarose HTL configuration.

In [ ]:
flat_htl_learning_stats_exports = [
    {
        "label": "Control learning_stats.csv for upd KO agarose-time analysis (flat HTL)",
        "command": """mkdir -p '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO' && python analyze.py -v '/media/Synology4/Yang Chen/2025-02-01/c[12]_*,/media/Synology4/Yang Chen/2025-02-01/c2[12]_*,/media/Synology4/Yang Chen/2025-02-0[12]/c3[12]_*,/media/Synology4/Yang Chen/2025-02-01/c4[12]_*' -f 0-9 --rmCC 5 --agarose && cp learning_stats.csv '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.ctrl.flat_htl.csv'""",
        "output_path": "/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.ctrl.flat_htl.csv",
    },
    {
        "label": "upd3 KO learning_stats.csv for agarose-time analysis (flat HTL)",
        "command": """mkdir -p '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO' && python analyze.py -v '/media/Synology4/Yang Chen/2025-01-1[267]/c[12]_*,/media/Synology4/Yang Chen/2025-01-1[267]/c3[12]_*' -f 0-9 --rmCC 5 --agarose && cp learning_stats.csv '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.mutant.flat_htl.csv'""",
        "output_path": "/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.mutant.flat_htl.csv",
    },
    {
        "label": "upd2+3 KO learning_stats.csv for agarose-time analysis (flat HTL)",
        "command": """mkdir -p '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO' && python analyze.py -v '/media/Synology4/Yang Chen/2025-01-1[267]/c2[12]_*,/media/Synology4/Yang Chen/2025-01-1[267]/c4[12]_*,/media/Synology4/Yang Chen/2025-01-22/c2[12]_*' -f 0-9 --rmCC 5 --agarose && cp learning_stats.csv '/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO/learning_stats.upd2+3_ko.mutant.flat_htl.csv'""",
        "output_path": "/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO/learning_stats.upd2+3_ko.mutant.flat_htl.csv",
    },
]

RUN_FLAT_HTL_LEARNING_STATS_EXPORTS = False
run_stage(
    "Flat HTL learning_stats.csv exports",
    flat_htl_learning_stats_exports,
    run=RUN_FLAT_HTL_LEARNING_STATS_EXPORTS,
)


### Flat HTL GraphPad CSV

This final stage exports the same per-fly reduction values:

`pre last 10m - T3 post last 10m`

The values are percentage-point reductions in percent time spent on agarose.

In [ ]:
flat_htl_graphpad_exports = [
    {
        "label": "Percent time on agarose reduction (flat HTL) -> GraphPad CSV",
        "command": "python scripts/export_graphpad_csv.py agarose-time --group Control=/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.ctrl.flat_htl.csv --group 'upd3 KO=/media/Synology4/Robert/JAK_STAT_upd_mutants/upd3_KO/learning_stats.upd3_ko.mutant.flat_htl.csv' --group 'upd2+3 KO=/media/Synology4/Robert/JAK_STAT_upd_mutants/upd2+3_KO/learning_stats.upd2+3_ko.mutant.flat_htl.csv' --section '% time over agarose (contact events begin when body center crosses agarose border)' --out exports/agarose_time_reduction_flat_htl_graphpad.csv",
        "output_path": "exports/agarose_time_reduction_flat_htl_graphpad.csv",
    },
]

RUN_FLAT_HTL_GRAPHPAD_EXPORT = False
run_stage(
    "Flat HTL reduction values -> GraphPad CSV",
    flat_htl_graphpad_exports,
    run=RUN_FLAT_HTL_GRAPHPAD_EXPORT,
)
